In [ ]:
import importlib
import sys

spec = importlib.util.spec_from_file_location("lib", "lib/__init__.py")
module_obj = importlib.util.module_from_spec(spec)
sys.modules["lib"] = module_obj
spec.loader.exec_module(module_obj)

from lib import PipelineConfig

In [ ]:
if "snakemake" in locals():
    feeder_outputs = snakemake.params.feeder_outputs
    no_feeder_outputs = snakemake.params.no_feeder_outputs
    input_paths = snakemake.input
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    scenario_name = snakemake.wildcards.scenario
    scenario_config = pipeline_config.scenarios[scenario_name]
else:
    raise Exception("This notebook is only snakemake-compatible for now")

In [ ]:
print("Values considered for the radius")
scenario_config.feeders.radiis

In [ ]:
print("Values considered for the frequency")
scenario_config.feeders.frequencies

In [ ]:
print("Values considered for the speeed")
scenario_config.feeders.speeds

In [ ]:
print("List of passed input paths")
feeder_outputs

Each Path is of the form /path/to/something/simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
- radius: the value of the radius parameter
- frequency: value of the frequency
- speed_index: index of the speed value in the speeds list displayed above (I didn't want to risk passing floats around)

Next we display the path to the outputs of the simulation corresponding to the scenario without feeder service
The variable can be none if no such simulation is provided

In [ ]:
no_feeder_outputs

# What to do next ?

In general we want to measure the relationship between the level of introduction of feeder services and how much better could the trips get relatively to existing public transport.

Some sanity checks, verify that all the scenarios have the same set of trips (person_id, person_trip_id, departure_time, origin, destination). If not, keep only the same ones while we investigate the source of the problem.

Parse the simulation outputs corresponding to each feeder settings and do some analysis.
Maybe the interesting KPI to consider here is the route cost of each trip. It is the one optimized directly by the routing algorithm so I think it makes sense. Here are the parameters:
- `rail_u_h`, `subway_u_h`, `bus_u_h`, `tram_u_h`, `other_u_h`: the marginal utility of time (in hours) spent in respective PT modes. The default value of these parameters is -7.
- `wait_u_h`: the marginal utility of time (in hours) spent waiting for public transport, no matter its mode.  The default value of this parameter is -6.
- `walk_u_h`: the markinal utility of time (in hours) spent walking to/from/between public transport legs.  Its default value is -7.
- `transfer_u`: the marginal utility of a transfer. Its default value is -1.

We can consider the whole chain including feeders (so also considering those transfers) and weighing time spent in a feeder service similarly to the bus. Instead of just keeping the total cost for the trip, we can also have the different components separated.

Then a sanity check we can have is to make sure that no trip sees its routing cost get worse before/after feeder.

Then we can have various graphs with this
- One line plot, feeder radius on the x-axis, total cost on the y axis. If multiple frequencies are considered, we put them in differently coloured lines. If multiple speeds are considered, we can used dashed lines.
- Same plot but instead of total routing cost, we can have the number of trips where the routing cost improves.
- Stacked bar plot, x axis for the feeder radius, y axis for the total cost, colors for the cost components, facet_col for frequency and facet_row for speed.
Then we can check the impact on the usage of existing PT systems (even though we should already kind of have an idea with the cost components)
- A plot showing the number of pt legs per pt mode (rail, subway, tram, feeder) in each setting.
- ...Other analyses